In [4]:
# Import required libraries
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import joblib
import os
import hashlib

In [5]:
def term_based_guardrail(question):
    """
    Use term-based approach to determine if a question is about Singapore history.
    
    Args:
        question (str): The user's question
    
    Returns:
        tuple: (bool indicating if question is relevant, string explanation)
    """
    # Lists of terms related to Singapore history
    sg_history_terms = [
    # General Singapore terms
    "singapore", "sg", "lion city", "little red dot", "singapura", "republic of singapore",
    
    # Historical figures
    "raffles", "sir stamford raffles", "lee kuan yew", "lky", "goh keng swee", 
    "lim bo seng", "s rajaratnam", "yusof ishak", "david marshall", "tan kah kee",
    "toh chin chye", "devan nair", "ong teng cheong", "wee kim wee", "benjamin sheares",
    "lim chin siong", "kwa geok choo", "lim yew hock", "tunku abdul rahman",
    
    # Colonial era
    "british colony", "straits settlements", "temasek", "east india company", 
    "crown colony", "colonial", "malaya", "farquhar", "john crawfurd", "penang",
    "melaka", "johor", "sultan hussein", "temenggong", "treaty of friendship",
    
    # Independence and nation-building
    "independence", "merger", "separation", "1965", "9 august", "malaysia",
    "people's action party", "pap", "self-government", "legislative assembly",
    "referendum", "federation", "city council", "merdeka", "national pledge",
    "national anthem", "majulah singapura", "national day", "ndp", 
    
    # Landmarks and locations
    "merlion", "fort canning", "botanic gardens", "chinatown", "little india", 
    "kampong glam", "bugis", "raffles hotel", "padang", "empress place", "city hall",
    "kallang", "jurong", "toa payoh", "changi", "sentosa", "bukit timah",
    "singapore river", "clarke quay", "boat quay", "queenstown", "mrt", "port of singapore",
    "psa", "keppel", "tanjong pagar", "sembawang", "bukit timah hill", "ford factory",
    
    # Events
    "japanese occupation", "syonan-to", "world war ii", "ww2", "battle of singapore",
    "coldstore", "operation coldstore", "konfrontasi", "malayan emergency", 
    "maria hertogh riots", "hock lee bus riots", "racial riots", "1964 riots",
    "communist insurgency", "macdonald house bombing", "sook ching", "surrender",
    "fall of singapore", "emergency", "internal security act", "isa",
    
    # Organizations and institutions
    "chinese chamber", "sclcc", "parliament", "asean", "housing development board", "hdb",
    "central provident fund", "cpf", "economic development board", "edb",
    "people's defence force", "saf", "singapore armed forces", "psa", "nus", "ntu",
    "nanyang university", "university of singapore", "raffles institution",
    "polytechnic", "jurong town corporation", "jtc", "ntuc", "trade union",
    
    # Cultural and social
    "dialect", "samsui", "peranakan", "straits chinese", "eurasian", "hawker",
    "kampong", "attap", "five foot way", "shophouse", "trishaw", "singlish",
    "national service", "ns", "multiculturalism", "bilingual policy", "education system",
    "public housing", "resettlement", "urban renewal", "speak mandarin campaign",
    
    # Development aspects
    "financial hub", "garden city", "water", "water agreements", "clean river",
    "industrialisation", "foreign investment", "currency board", "monetary authority",
    "public housing", "economic miracle", "asian tiger", "smart nation"
]
    
    # Lists of terms likely indicating non-history questions
    non_history_terms = [
    # Food and cooking
    "recipe", "cook", "bake", "meal", "dinner", "breakfast", "lunch", "ingredient",
    "steak", "chicken", "fry", "grill", "roast", "microwave", "oven", "stove",
    "delicious", "taste", "flavor", "cuisine", "menu", "restaurant", "chef",
    "kitchen", "baking", "chocolate", "cake", "cookie", "dessert", "appetizer",
    
    # Sports
    "football", "soccer", "basketball", "tennis", "golf", "swimming", "volleyball",
    "baseball", "hockey", "cricket", "rugby", "formula one", "f1", "olympics",
    "world cup", "championship", "tournament", "match", "game", "score", "team",
    "ronaldo", "messi", "lebron", "federer", "serena", "bolt", "phelps", "hamiltom",
    "premier league", "la liga", "nba", "nfl", "mlb", "fitness", "workout", "gym",
    
    # Entertainment
    "movie", "film", "series", "tv show", "netflix", "youtube", "hulu", "disney+",
    "actor", "actress", "director", "hollywood", "bollywood", "drama", "comedy",
    "thriller", "horror", "sci-fi", "animation", "documentary", "stream", "binge",
    "episode", "season", "premiere", "award", "oscar", "emmy", "grammy", "music",
    "song", "album", "artist", "band", "concert", "festival", "playlist", "spotify",
    "game", "gaming", "playstation", "xbox", "nintendo", "steam", "twitch",
    
    # Technology
    "phone", "computer", "laptop", "tablet", "gadget", "device", "app", "software",
    "hardware", "iphone", "android", "samsung", "apple", "google", "microsoft",
    "programming", "code", "developer", "website", "internet", "wifi", "bluetooth",
    "smart home", "ai", "artificial intelligence", "machine learning", "algorithm",
    "data", "cloud", "server", "cybersecurity", "hack", "virus", "malware",
    "update", "download", "upload", "browser", "search engine", "social media",
    
    # Finance and business
    "stock market", "investment", "bitcoin", "cryptocurrency", "ethereum", "dogecoin",
    "forex", "trading", "stock", "share", "dividend", "mutual fund", "etf", "ira",
    "401k", "retirement", "savings", "loan", "mortgage", "interest rate", "bank",
    "credit card", "debit card", "credit score", "debt", "budget", "expense",
    "income", "profit", "loss", "startup", "entrepreneur", "business plan",
    "marketing", "sales", "customer", "product", "service", "partnership",
    
    # Lifestyle and health
    "diet", "exercise", "fitness", "workout", "gym", "yoga", "pilates", "meditation",
    "mindfulness", "health", "wellness", "nutrition", "vitamin", "supplement",
    "weight loss", "muscle gain", "cardio", "strength training", "running",
    "jogging", "cycling", "marathon", "bodybuilding", "crossfit", "keto",
    "vegan", "vegetarian", "gluten-free", "organic", "natural", "sleep",
    "stress", "anxiety", "depression", "therapy", "psychology", "self-help",
    
    # Travel and leisure
    "vacation", "trip", "travel", "tourism", "tourist", "hotel", "resort",
    "flight", "airline", "booking", "reservation", "passport", "visa",
    "sightseeing", "landmark", "beach", "mountain", "hiking", "camping",
    "backpacking", "cruise", "road trip", "destination", "itinerary",
    "souvenir", "photography", "camera", "lens", "photo", "picture",
    
    # Home and auto
    "clean", "cleaning", "declutter", "organize", "decoration", "furniture",
    "renovation", "repair", "maintenance", "garden", "lawn", "plants",
    "flowers", "car", "vehicle", "automobile", "tire", "engine", "transmission",
    "brake", "oil change", "warranty", "insurance", "dealership", "fuel",
    "gas", "electric vehicle", "hybrid", "charging station", "theater system",
    
    # Relationships and social
    "relationship", "dating", "marriage", "partner", "spouse", "boyfriend",
    "girlfriend", "husband", "wife", "divorce", "breakup", "family", "children",
    "parenting", "baby", "toddler", "teenager", "elder", "friendship", "social",
    "communication", "conflict", "resolution", "therapy", "counseling", "advice",
    
    # Education and career
    "study", "student", "teacher", "professor", "school", "college", "university",
    "degree", "diploma", "certificate", "course", "class", "lecture", "seminar",
    "workshop", "education", "learning", "exam", "test", "grade", "gpa", "scholarship",
    "career", "job", "employment", "resume", "cv", "interview", "application",
    "salary", "wage", "promotion", "office", "workplace", "colleague", "boss",
    "productivity", "time management", "project management", "remote work",
    
    # Miscellaneous
    "joke", "funny", "humor", "meme", "pet", "dog", "cat", "fish", "bird",
    "veterinarian", "adopt", "breed", "training", "grooming", "weather", "forecast",
    "temperature", "rain", "snow", "sunny", "cloudy", "storm", "hurricane",
    "climate", "environment", "pollution", "recycling", "sustainable", "organic",
    "coffee", "tea", "alcohol", "beer", "wine", "cocktail", "drink", "beverage",
    "crocodile", "alligator", "animal", "wildlife", "zoo", "safari", "nature"
]
    
    # Normalize the question
    question_lower = question.lower()
    
    # Check for Singapore history terms
    sg_terms_found = [term for term in sg_history_terms if term in question_lower]
    
    # Check for non-history terms
    non_sg_terms_found = [term for term in non_history_terms if term in question_lower]
    
    # Decision logic
    if len(sg_terms_found) > 0 and len(non_sg_terms_found) == 0:
        # Contains Singapore history terms but no non-history terms
        return True, f"Found relevant terms: {', '.join(sg_terms_found)}"
    elif len(sg_terms_found) > 0 and len(non_sg_terms_found) > 0:
        # Contains both types of terms - need to make a decision
        if len(sg_terms_found) > len(non_sg_terms_found):
            return True, f"More history terms ({len(sg_terms_found)}) than non-history terms ({len(non_sg_terms_found)})"
        else:
            return False, f"Contains non-history terms: {', '.join(non_sg_terms_found)}"
    elif len(sg_terms_found) == 0 and len(non_sg_terms_found) > 0:
        # Contains only non-history terms
        return False, f"Contains non-history terms: {', '.join(non_sg_terms_found)}"
    else:
        # No recognized terms at all - default to rejecting
        return False, "No recognized Singapore history terms found in your question"

In [6]:
def improved_chatbot_response(question: str) -> str:
    """
    Improved chatbot response using term-based guardrail.
    
    Args:
        question (str): User's question
        
    Returns:
        str: Response to the user
    """
    is_relevant, reason = term_based_guardrail(question)
    
    if is_relevant:
        # Only history-related questions get processed
        return f"Processing your question about Singapore's history... (Reason: {reason})"
    else:
        # STRICT BLOCKING of unrelated questions
        return f"I'm sorry, but I can only answer questions related to Singapore's history. (Reason: {reason})"

In [7]:
# Example test cases
test_questions = [
    "Who was Sir Stamford Raffles?",
    "How do I cook a steak?",
    "Tell me about the Japanese Occupation in Singapore.",
    "What is the best way to invest in stocks?",
    "Explain Singapore's role in World War II.",
    "Can you recommend a good movie?",
    # Additional edge cases
    "Who is Ronaldo?",
    "What is sundal?",
    "Was Singapore affected by World War II?",
    "Tell me about the history of HDB flats"
]

# Generate responses for test cases
for question in test_questions:
    response = improved_chatbot_response(question)
    print(f"Q: {question}\nA: {response}\n")

Q: Who was Sir Stamford Raffles?
A: Processing your question about Singapore's history... (Reason: Found relevant terms: raffles, sir stamford raffles)

Q: How do I cook a steak?
A: I'm sorry, but I can only answer questions related to Singapore's history. (Reason: Contains non-history terms: cook, steak, tea)

Q: Tell me about the Japanese Occupation in Singapore.
A: Processing your question about Singapore's history... (Reason: Found relevant terms: singapore, japanese occupation)

Q: What is the best way to invest in stocks?
A: I'm sorry, but I can only answer questions related to Singapore's history. (Reason: Contains non-history terms: stock)

Q: Explain Singapore's role in World War II.
A: Processing your question about Singapore's history... (Reason: More history terms (2) than non-history terms (1))

Q: Can you recommend a good movie?
A: I'm sorry, but I can only answer questions related to Singapore's history. (Reason: Contains non-history terms: movie)

Q: Who is Ronaldo?
A: 

In [8]:
# Input/Output schema validation using Pydantic (optional)
from pydantic import BaseModel, ValidationError
from typing import Any

# Define expected input schema
class InputSchema(BaseModel):
    data: Any  # Adjust type according to expected input

# Define expected output schema
class OutputSchema(BaseModel):
    result: Any  # Adjust type according to expected output

def validate_input(input_data):
    try:
        validated_data = InputSchema(**input_data)
        return validated_data
    except ValidationError as e:
        print("Input validation failed:", e)
        return None

def validate_output(output_data):
    try:
        validated_result = OutputSchema(**output_data)
        return validated_result
    except ValidationError as e:
        print("Output validation failed:", e)
        return None

# Install Dependencies


In [9]:
!pip install -q unsloth
!pip install -q langchain
!pip install -q chromadb
!pip install -q pypdf
!pip install -U langchain-community
!pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.3/196.3 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 32.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.1/253.1 MB 6.7 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.9/318.9 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.5/129.5 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 MB 40.5 MB/s eta 0:00:00:00:010:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.7/766.7 MB 2.2 MB/s eta 0:00:000:00:010:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 97.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6

In [15]:
!pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://download.pytorch.org/whl/cu118


# Import Libraries

In [16]:
# Import Libraries for RAG and model loading
from unsloth import FastLanguageModel
import os
import torch
from torch.utils.data import Dataset
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from langchain.schema import Document
from transformers import AutoTokenizer, Trainer, TrainingArguments
from peft import PeftModel
import gc

In [17]:
# Clean up memory
gc.collect()
torch.cuda.empty_cache()

# Model configuration
max_seq_length = 2048
dtype = torch.float16  # T4 works best with float16
load_in_4bit = True  # 4-bit quantization for memory efficiency

In [18]:
max_seq_length = 2048
dtype = torch.float16  # T4 works best with float16
load_in_4bit = True  # 4-bit quantization for memory efficiency

## PDF path for RAG


In [19]:
pdf_path = "/kaggle/input/sg-history-090325"  # Replace with your actual path

## Load The Pre-Trained Model (Need to add the checkpoint here)


In [20]:
from unsloth import FastLanguageModel
from peft import PeftModel
import torch

# 🔹 Path to your fine-tuned checkpoint
checkpoint_path = "/kaggle/input/phi-trained-sg-hist/pytorch/default/1/phi3_trained_model/checkpoint-800"  # Update this to your actual path

print(f"Loading fine-tuned model from {checkpoint_path}...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=checkpoint_path,  # Load from checkpoint instead of pre-trained model
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print("✅ Fine-tuned model loaded successfully!")



Loading fine-tuned model from /kaggle/input/phi-trained-sg-hist/pytorch/default/1/phi3_trained_model/checkpoint-800...
==((====))==  Unsloth 2025.3.17: Fast Llama patching. Transformers: 4.49.0.
   \\   /|    Tesla P100-PCIE-16GB. Num GPUs = 1. Max memory: 15.888 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu118. CUDA: 6.0. CUDA Toolkit: 11.8. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.26G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/140 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.37k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

Unsloth 2025.3.17 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


✅ Fine-tuned model loaded successfully!


# Stop test running here - 090325 test


## Using Langchain to process the pdf for Rag

In [21]:
# Function to find and process PDF files for RAG
def find_pdf_files(directory):
    pdf_files = []
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.lower().endswith('.pdf'):
                pdf_files.append(os.path.join(root, file))
    return pdf_files

def load_or_create_vectorstore(directory, embedding_model, force_rebuild=False):
    # Define a persistent directory with a hash based on the directory path
    dir_hash = hashlib.md5(directory.encode()).hexdigest()[:8]
    persist_directory = f"./chroma_db_{dir_hash}"
    
    if not force_rebuild and os.path.exists(persist_directory) and os.listdir(persist_directory):
        print(f"Loading existing vector store from {persist_directory}...")
        return Chroma(
            persist_directory=persist_directory,
            embedding_function=embedding_model
        )
    
    print(f"Creating new vector store from {directory}...")
    # Process PDFs and create text chunks
    text_content = process_multiple_pdfs(directory)
    
    if not text_content:
        raise ValueError("No content extracted from PDFs to build vector store.")
        
    # Create documents with metadata
    documents = []
    for i, chunk in enumerate(text_content):
        doc = Document(
            page_content=chunk,
            metadata={"source": f"chunk_{i}"}
        )
        documents.append(doc)
    
    # Create and persist vector store
    vector_store = Chroma.from_documents(
        documents=documents,
        embedding=embedding_model,
        persist_directory=persist_directory
    )
    # Explicitly persist the vector store
    vector_store.persist()
    print(f"Vector store created and persisted at {persist_directory}")
    return vector_store

def process_multiple_pdfs(directory):
    print(f"Searching for PDFs in: {directory}")
    pdf_files = find_pdf_files(directory)

    if not pdf_files:
        print(f"No PDF files found in {directory}")
        return []

    print(f"Found {len(pdf_files)} PDF files.")
    all_text_chunks = []

    for pdf_file in pdf_files:
        try:
            print(f"Processing: {pdf_file}")
            pdf_loader = PyPDFLoader(pdf_file)
            documents = pdf_loader.load()

            if not documents:
                print(f"No content loaded from {pdf_file}")
                continue

            print(f"Loaded {len(documents)} pages from {pdf_file}")

            text_splitter = RecursiveCharacterTextSplitter(
                chunk_size=512,
                chunk_overlap=50,
                separators=["\n\n", "\n", ".", " ", ""]
            )
            text_chunks = text_splitter.split_documents(documents)
            print(f"Split into {len(text_chunks)} text chunks")

            text_content = [chunk.page_content for chunk in text_chunks]
            all_text_chunks.extend(text_content)

            del documents
            del text_chunks
            gc.collect()

        except Exception as e:
            print(f"Error processing {pdf_file}: {str(e)}")
            import traceback
            traceback.print_exc()

    print(f"Total text chunks from all PDFs: {len(all_text_chunks)}")
        
    return all_text_chunks

In [22]:
def create_instruction_format(text):
    return f"<|system|>\nYou are an AI assistant that provides helpful, accurate information based on your training.\n<|user|>\nAnalyze and summarize this text passage: {text}\n<|assistant|>\nHere's my analysis and summary of the passage:"

class TextDataset(Dataset):
    def __init__(self, tokenized_texts):
        self.input_ids = tokenized_texts["input_ids"]
        self.attention_mask = tokenized_texts["attention_mask"]

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'labels': self.input_ids[idx].clone()
        }

# Load the pre-trained model
def load_fine_tuned_model(checkpoint_path, max_seq_length=2048, dtype=torch.float16, load_in_4bit=True):
    print(f"Loading fine-tuned model from {checkpoint_path}...")
    
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=checkpoint_path,  # Load from checkpoint instead of pre-trained model
        max_seq_length=max_seq_length,
        dtype=dtype,
        load_in_4bit=load_in_4bit,
    )
    
    print("✅ Fine-tuned model loaded successfully!")
    return model, tokenizer

# Path to your fine-tuned checkpoint
checkpoint_path = "/kaggle/input/phi-trained-sg-hist/pytorch/default/1/phi3_trained_model/checkpoint-800"

# Load the model
model, tokenizer = load_fine_tuned_model(
    checkpoint_path=checkpoint_path,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit
)

# Set up the embedding model
embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=embedding_model_name)

# Set to True to force rebuilding the vector store and retraining the classifier
FORCE_REBUILD = False

# Create or load the vector store
vector_store = load_or_create_vectorstore(
    directory=pdf_path,
    embedding_model=embeddings,
    force_rebuild=FORCE_REBUILD
)

Loading fine-tuned model from /kaggle/input/phi-trained-sg-hist/pytorch/default/1/phi3_trained_model/checkpoint-800...
==((====))==  Unsloth 2025.3.17: Fast Llama patching. Transformers: 4.49.0.
   \\   /|    Tesla P100-PCIE-16GB. Num GPUs = 1. Max memory: 15.888 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu118. CUDA: 6.0. CUDA Toolkit: 11.8. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
✅ Fine-tuned model loaded successfully!


<ipython-input-22-d2ff95c95f81>:46: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=embedding_model_name)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Creating new vector store from /kaggle/input/sg-history-090325...
Searching for PDFs in: /kaggle/input/sg-history-090325
Found 33 PDF files.
Processing: /kaggle/input/sg-history-090325/Dataset/CHINATOWNS IN A GLOBALIZING SOUTHEAST ASIA ( etc.) (Z-Library).pdf
Loaded 175 pages from /kaggle/input/sg-history-090325/Dataset/CHINATOWNS IN A GLOBALIZING SOUTHEAST ASIA ( etc.) (Z-Library).pdf
Split into 4 text chunks
Processing: /kaggle/input/sg-history-090325/Dataset/To_Catch_a_Tartar_A_Dissident_in_Lee_Kuan_Yews_Prison_Francis_Seow.pdf
Loaded 329 pages from /kaggle/input/sg-history-090325/Dataset/To_Catch_a_Tartar_A_Dissident_in_Lee_Kuan_Yews_Prison_Francis_Seow.pdf
Split into 1783 text chunks
Processing: /kaggle/input/sg-history-090325/Dataset/Comet_in_our_sky_Lim_Chin_Siong_in_history_TAN_Jing_Quee,_JOMO_K.pdf
Loaded 192 pages from /kaggle/input/sg-history-090325/Dataset/Comet_in_our_sky_Lim_Chin_Siong_in_history_TAN_Jing_Quee,_JOMO_K.pdf
Split into 984 text chunks
Processing: /kaggle/inp

<ipython-input-21-4b8dd5027551>:45: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vector_store.persist()


# Test Model Normal Inference


In [23]:
# Normal Inference Function
def ask_question(question):
    prompt = f"<|system|>\n You are a LLM pretrained with information about singapore. \n<|user|>\n{question}\n<|assistant|>\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=512)
    full_response = tokenizer.decode(outputs[0], skip_special_tokens=False)
    
    # Extract just the first assistant response
    # Find the position of <|assistant|> tag
    assistant_tag_pos = full_response.find("<|assistant|>")
    if assistant_tag_pos != -1:
        # Extract text after the first assistant tag
        assistant_response = full_response[assistant_tag_pos + len("<|assistant|>"):].strip()
        
        # If there's a USER tag, cut off at that point
        user_tag_pos = assistant_response.find("##### USER:")
        if user_tag_pos != -1:
            assistant_response = assistant_response[:user_tag_pos].strip()
        
        # If there's another assistant tag, cut off at that point too
        next_assistant_tag = assistant_response.find("<|assistant|>")
        if next_assistant_tag != -1:
            assistant_response = assistant_response[:next_assistant_tag].strip()
            
        return assistant_response
    else:
        return "Error extracting response"

In [24]:
# Import required libraries for vector store
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma

# Set up the embedding model
embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=embedding_model_name)

# Load the existing vector store
print("Loading existing vector store...")
vector_store = Chroma(
    persist_directory="/kaggle/working/chroma_db_cdac0f11",  # Update with your actual path
    embedding_function=embeddings
)
print("Vector store loaded successfully!")

Loading existing vector store...
Vector store loaded successfully!


<ipython-input-24-f73e5e0a34d2>:11: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vector_store = Chroma(


In [25]:
# RAG-enhanced question answering function
def rag_ask_question(question, top_k=3):
    # First, check if the question is related to Singapore history
    is_relevant, reason = term_based_guardrail(question)
    
    if not is_relevant:
        return f"I'm sorry, but I can only answer questions related to Singapore's history. (Reason: {reason})"
    
    # If the question is related to Singapore history, proceed with RAG
    # Step 1: Retrieve relevant documents
    retrieved_docs = vector_store.similarity_search(question, k=top_k)
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])
    
    # Step 2: Create RAG-enhanced prompt
    rag_prompt = f"""<|system|>
You are a specialized AI assistant focused on Singapore's history. You have been trained on comprehensive historical resources about Singapore's history, culture, and development.

Here is some relevant information to help answer the user's question:
{context}

Answer the question based on the provided information. If the information is not sufficient to answer the question confidently, acknowledge the limitations. Provide accurate, educational responses that help users better understand Singapore's rich historical narrative, based on documents you have retrieved.
<|user|>
{question}
<|assistant|>
"""
    
    # Step 3: Generate response
    inputs = tokenizer(rag_prompt, return_tensors="pt").to(model.device)
    
    # Manage sequence length to avoid exceeding model's context window
    if inputs['input_ids'].shape[1] > max_seq_length - 512:  # Reserve space for generation
        # Truncate if too long
        inputs = tokenizer(
            rag_prompt, 
            truncation=True, 
            max_length=max_seq_length - 512, 
            return_tensors="pt"
        ).to(model.device)
    
    # Generate response
    outputs = model.generate(**inputs, max_new_tokens=512)
    full_response = tokenizer.decode(outputs[0], skip_special_tokens=False)
    
    # Extract just the assistant's response
    assistant_tag_pos = full_response.find("<|assistant|>")
    if assistant_tag_pos != -1:
        assistant_response = full_response[assistant_tag_pos + len("<|assistant|>"):].strip()
        
        # Clean up response by removing any subsequent tags
        user_tag_pos = assistant_response.find("<|user|>")
        if user_tag_pos != -1:
            assistant_response = assistant_response[:user_tag_pos].strip()
        
        system_tag_pos = assistant_response.find("<|system|>")
        if system_tag_pos != -1:
            assistant_response = assistant_response[:system_tag_pos].strip()
        
        return assistant_response
    else:
        return "Error extracting response"

In [ ]:
# Interactive question answering loop
def start_interactive_session():
    print("\nSingapore History RAG System is ready!")
    print("Ask questions about Singapore's history, or type 'exit' to quit.")
    while True:
        user_question = input("\nYour question: ")
        if user_question.lower() == "exit":
            break
        try:
            response = rag_ask_question(user_question)
            
            # Clean up any repeated tokens, special tags, or unusual characters
            # First handle special tokens
            for token in ["<|assistant|>", "<|end|>", "<|user|>", "<|system|>"]:
                if token in response:
                    response = response.split(token)[0].strip()
            
            # Handle long repeating characters (like zeros)
            import re
            response = re.sub(r'([0-9])\1{10,}', '', response)
            
            # Remove any non-text content at the end
            response = re.sub(r'[^\w\s.,;:!?()-]$', '', response)
            
            print("\nAnswer:")
            print(response)
        except Exception as e:
            print(f"An error occurred: {e}")
            import traceback
            traceback.print_exc()

# Start the interactive session
if __name__ == "__main__":
    start_interactive_session()


Singapore History RAG System is ready!
Ask questions about Singapore's history, or type 'exit' to quit.



Your question:  What is operation Coldstore?



Answer:
Operation Coldstore was a massive preventative security action in February 1963 that detained 133 people: nine fully identified MCP ‘underground’ operatives who were believed to be involved in subversive activities. This operation was a response to the perceived threat of communist infiltration and subversion, and was part of a broader effort to maintain social and political stability in Singapore during a period of significant political and social change.

The operation was controversial and has been the subject of much debate and analysis. Some have argued that it was a necessary measure to protect Singapore's security and stability, while others have criticized it as an overreach of government power and an infringement on civil liberties.

In summary, Operation Coldstore was a significant event in Singapore's history, reflecting the tensions and challenges of the time. It remains a topic of discussion and analysis, highlighting the complexities of balancing security and civ


Your question:  Is Jit lin a commie



Answer:
I'm sorry, but I can only answer questions related to Singapore's history. (Reason: No recognized Singapore history terms found in your question)



Your question:  What is sundal



Answer:
I'm sorry, but I can only answer questions related to Singapore's history. (Reason: No recognized Singapore history terms found in your question)
